In [ ]:
import xarray as xr
import numpy as np
import os

# =====================================================
# CONFIGURATION
# =====================================================
cleaned_files = [
    r"E:\ML Project\WOSC Project\EarthFormer\Cleaned_Netcdf\cleaned_9_Yaas.nc"
]

vars_to_use = ["to", "so", "zo"]

OUTPUT_DIR = r"E:\ML Project\WOSC Project\WOSC Dynamic"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAT_MIN, LAT_MAX = 18.3, 19.3
LON_MIN, LON_MAX = 85.32, 86.32

DEPTH_FULL = np.array([
    0,5,10,15,20,25,30,35,40,45,50,55,60,65,70,80,90,100,125,150,175,200,225,250,275,
    300,350,400,450,500,550,600,700
], dtype=np.float32)
depth_sel = DEPTH_FULL[DEPTH_FULL <= 100]

# =====================================================
# Cyclone windows 
# =====================================================
cyclone_windows = {
    9: {"pre": ("2021-05-13","2021-05-22"), "during": ("2021-05-23","2021-05-27")}
}

# =====================================================
# MAIN PIPELINE
# =====================================================
for i, nc_path in enumerate(cleaned_files, start=9):  # 9 corresponds to Yaas
    ds = xr.open_dataset(nc_path)
    name = ds.attrs.get("cyclone_name", f"cyclone_{i}")
    print(f"\nProcessing cyclone: {name}")

    # ---- Crop spatially ----
    lat_vals = ds.latitude.values
    lon_vals = ds.longitude.values
    lat_idx = np.where((lat_vals >= LAT_MIN) & (lat_vals <= LAT_MAX))[0]
    lon_idx = np.where((lon_vals >= LON_MIN) & (lon_vals <= LON_MAX))[0]
    ds = ds.sel(latitude=lat_vals[lat_idx], longitude=lon_vals[lon_idx])
    H_crop, W_crop = len(lat_idx), len(lon_idx)

    # ---- Crop depths ----
    ds = ds.sel(depth=depth_sel)
    D = len(depth_sel)

    # ---- Split pre and during ----
    win = cyclone_windows[i]
    ds_before = ds.sel(time=slice(*win["pre"]))
    ds_during = ds.sel(time=slice(*win["during"]))

    ds_during = ds_during.isel(time=slice(0, 5))

    T_during = len(ds_during.time)
    time_vals = ds_during.time.values

    # ---- Pre-cyclone mean ----
    pre_mean = {v: ds_before[v].mean(dim="time").values for v in vars_to_use}

    # ---- Build 5D pre-mean array for NetCDF ----
    # Shape: (time, depth, lat, lon, var)
    X_pre_mean = np.stack([pre_mean[v] for v in vars_to_use], axis=-1)  # (D,H,W,V)
    X_pre_mean = np.repeat(X_pre_mean[None, ...], T_during, axis=0)     # (T,D,H,W,V)

    # ---- Create xarray Dataset ----
    ds_pre_mean = xr.Dataset(
        {v: (["time", "depth", "latitude", "longitude"], X_pre_mean[..., idx])
         for idx, v in enumerate(vars_to_use)},
        coords={
            "time": time_vals,
            "depth": depth_sel,
            "latitude": lat_vals[lat_idx],
            "longitude": lon_vals[lon_idx]
        }
    )

    # ---- Save as NetCDF ----
    netcdf_path = os.path.join(OUTPUT_DIR, f"{name}_pre_mean.nc")
    ds_pre_mean.to_netcdf(netcdf_path)
    print(f"Saved pre-mean NetCDF: {name} at {netcdf_path}")





Processing cyclone: Yaas
Saved pre-mean NetCDF for Yaas at E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_pre_mean.nc

ALL CYCLONES PROCESSED CORRECTLY


In [ ]:
import numpy as np
import xarray as xr
import os

# =====================================================
# CONFIG
# =====================================================

Y_PRED_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Y_pred.npy"
Y_TRUE_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Y_true.npy"
PRE_MEAN_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_pre_mean.nc"

OUTPUT_DIR = r"E:\ML Project\WOSC Project\WOSC"

vars_to_use = ["to", "so", "zo"]
N_VARS = len(vars_to_use)

# =====================================================
# STEP 1 LOAD ARRAYS
# =====================================================

Y_pred = np.load(Y_PRED_PATH)
Y_true = np.load(Y_TRUE_PATH)

print("Loaded arrays:")
print("Y_pred shape:", Y_pred.shape)
print("Y_true shape:", Y_true.shape)

# =====================================================
# STEP 2 LOAD PRE-MEAN NETCDF (FOR SHAPE + COORDS)
# =====================================================

pre_ds = xr.open_dataset(PRE_MEAN_PATH)

time_vals = pre_ds.time.values
depth_vals = pre_ds.depth.values
lat_vals = pre_ds.latitude.values
lon_vals = pre_ds.longitude.values

T = len(time_vals)
D = len(depth_vals)
H = len(lat_vals)
W = len(lon_vals)

print("\nDimensions from pre-mean:")
print("T, D, H, W =", T, D, H, W)
print("\nPre-mean dataset structure:")
print(pre_ds)


Loaded arrays:
Y_pred shape: (1, 5, 8, 8, 90)
Y_true shape: (1, 5, 8, 8, 90)

Dimensions from pre-mean:
T, D, H, W = 5 18 8 8

Pre-mean dataset structure:
<xarray.Dataset> Size: 138kB
Dimensions:    (time: 5, depth: 18, latitude: 8, longitude: 8)
Coordinates:
  * time       (time) datetime64[ns] 40B 2021-05-23 2021-05-24 ... 2021-05-27
  * depth      (depth) float32 72B 0.0 5.0 10.0 15.0 ... 70.0 80.0 90.0 100.0
  * latitude   (latitude) float32 32B 18.31 18.44 18.56 ... 18.94 19.06 19.19
  * longitude  (longitude) float32 32B 85.44 85.56 85.69 ... 86.06 86.19 86.31
Data variables:
    to         (time, depth, latitude, longitude) float64 46kB ...
    so         (time, depth, latitude, longitude) float64 46kB ...
    zo         (time, depth, latitude, longitude) float64 46kB ...


In [ ]:
import numpy as np

# =====================================================
# CONFIG
# =====================================================

PRED_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Y_pred.npy"
TRUE_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Y_true.npy"
PRE_MEAN_PATH = r"E:\ML Project\WOSC Project\WOSC_Dynamic\Yaas_pre_mean.nc"

# Physical dimensions (from your pre-mean netcdf)
D = 18          # number of depth levels
V = 3          # number of physical variables (to, so, ugo, vgo, zo)
V_PLUS_FORCING = 5   # 5 variables + 1 static forcing

# =====================================================
# LOAD ARRAYS
# =====================================================

Y_pred = np.load(PRED_PATH)
Y_true = np.load(TRUE_PATH)

print("Loaded arrays:")
print("Y_pred shape:", Y_pred.shape)
print("Y_true shape:", Y_true.shape)

# Expected shape:
# (1, 5, 8, 8, 105)

# =====================================================
# STEP 1 — Remove batch dimension
# =====================================================

Y_pred = Y_pred[0]   # (T, H, W, 105)
Y_true = Y_true[0]

print("\nAfter removing batch:")
print("Y_pred:", Y_pred.shape)

# =====================================================
# STEP 2 — Remove depth channel
# =====================================================

physical_channels = D * V_PLUS_FORCING   # 15 * 6 = 90

Y_pred_phys = Y_pred[..., :physical_channels]
Y_true_phys = Y_true[..., :physical_channels]

print("\nAfter removing depth channel:")
print("Y_pred_phys:", Y_pred_phys.shape)

# Should be:
# (T, H, W, 90)

# =====================================================
# STEP 3 — Unflatten back to 5D
# =====================================================

T, H, W, _ = Y_pred_phys.shape

# Reshape to (T, H, W, D, V+1)
Y_pred_5d = Y_pred_phys.reshape(T, H, W, D, V_PLUS_FORCING)
Y_true_5d = Y_true_phys.reshape(T, H, W, D, V_PLUS_FORCING)

# Reorder to (T, D, H, W, V+1)
Y_pred_5d = np.transpose(Y_pred_5d, (0, 3, 1, 2, 4))
Y_true_5d = np.transpose(Y_true_5d, (0, 3, 1, 2, 4))

print("\nAfter unflattening:")
print("Y_pred_5d:", Y_pred_5d.shape)

# Expected:
# (5, 15, 8, 8, 6)

# =====================================================
# STEP 4 — Remove static forcing channel
# =====================================================

Y_pred_anom = Y_pred_5d[..., :V]
Y_true_anom = Y_true_5d[..., :V]

print("\nAfter removing forcing:")
print("Y_pred_anom:", Y_pred_anom.shape)

# Final expected shape:
# (5, 15, 8, 8, 5)

print("Final anomaly shape:", Y_pred_anom.shape)

Loaded arrays:
Y_pred shape: (1, 5, 8, 8, 90)
Y_true shape: (1, 5, 8, 8, 90)

After removing batch:
Y_pred: (5, 8, 8, 90)

After removing depth channel:
Y_pred_phys: (5, 8, 8, 90)

After unflattening:
Y_pred_5d: (5, 18, 8, 8, 5)

After removing forcing:
Y_pred_anom: (5, 18, 8, 8, 3)

✅Done until Step 4
Final anomaly shape: (5, 18, 8, 8, 3)


In [ ]:
import numpy as np
import xarray as xr
import os

# =====================================================
# CONFIG
# =====================================================

PREMEAN_PATH = r"E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_pre_mean.nc"
OUTPUT_DIR = r"E:\ML Project\WOSC Project\WOSC Dynamic"
os.makedirs(OUTPUT_DIR, exist_ok=True)

vars_to_use = ["to", "so", "zo"]

# =====================================================
# STEP 1 : Load Pre-Mean NetCDF
# =====================================================

ds_pre = xr.open_dataset(PREMEAN_PATH)

print("Pre-mean dataset:")
print(ds_pre)

# Extract coordinates
time_vals = ds_pre.time.values
depth_vals = ds_pre.depth.values
lat_vals = ds_pre.latitude.values
lon_vals = ds_pre.longitude.values

# =====================================================
# STEP 2 : Convert pre-mean to numpy array
# =====================================================

# Stack variables to match anomaly shape
pre_mean_np = np.stack(
    [ds_pre[v].values for v in vars_to_use],
    axis=-1
)

print("Pre-mean shape:", pre_mean_np.shape)

# Should be:
# (5, 15, 8, 8, 5)

# =====================================================
# STEP 3 : Convert anomalies to actual values
# =====================================================

# STEP 3A : Inverse Global Scaling (Unscale)


# Global scaler values (physical variables only)
global_means = np.array([26.909142, 33.502992, 0.644818])
global_stds  = np.array([2.935716, 1.270493, 0.210552])

# Reshape for broadcasting: (1,1,1,1,3)
global_means = global_means.reshape(1, 1, 1, 1, -1)
global_stds  = global_stds.reshape(1, 1, 1, 1, -1)

# Inverse transform
Y_pred_unscaled = Y_pred_anom * global_stds
Y_true_unscaled = Y_true_anom * global_stds 

print("Unscaled shape:", Y_pred_unscaled.shape)

# STEP 3B : Add Pre-Mean to Recover Physical Values

Y_pred_actual = Y_pred_unscaled + pre_mean_np
Y_true_actual = Y_true_unscaled + pre_mean_np

print("Actual prediction shape:", Y_pred_actual.shape)

# =====================================================
# STEP 4 : Create NetCDF for Predictions
# =====================================================

ds_pred = xr.Dataset(
    {
        var: (["time", "depth", "latitude", "longitude"],
              Y_pred_actual[..., i])
        for i, var in enumerate(vars_to_use)
    },
    coords={
        "time": time_vals,
        "depth": depth_vals,
        "latitude": lat_vals,
        "longitude": lon_vals
    }
)

pred_path = os.path.join(OUTPUT_DIR, "Yaas_Predicted_Actual.nc")
ds_pred.to_netcdf(pred_path)

print(f"\n Saved predicted NetCDF at: {pred_path}")

# =====================================================
# STEP 5 : Create NetCDF for True Values
# =====================================================

ds_true = xr.Dataset(
    {
        var: (["time", "depth", "latitude", "longitude"],
              Y_true_actual[..., i])
        for i, var in enumerate(vars_to_use)
    },
    coords={
        "time": time_vals,
        "depth": depth_vals,
        "latitude": lat_vals,
        "longitude": lon_vals
    }
)

true_path = os.path.join(OUTPUT_DIR, "Yaas_True_Actual.nc")
ds_true.to_netcdf(true_path)

print(f" Saved true NetCDF at: {true_path}")


Pre-mean dataset:
<xarray.Dataset> Size: 138kB
Dimensions:    (time: 5, depth: 18, latitude: 8, longitude: 8)
Coordinates:
  * time       (time) datetime64[ns] 40B 2021-05-23 2021-05-24 ... 2021-05-27
  * depth      (depth) float32 72B 0.0 5.0 10.0 15.0 ... 70.0 80.0 90.0 100.0
  * latitude   (latitude) float32 32B 18.31 18.44 18.56 ... 18.94 19.06 19.19
  * longitude  (longitude) float32 32B 85.44 85.56 85.69 ... 86.06 86.19 86.31
Data variables:
    to         (time, depth, latitude, longitude) float64 46kB ...
    so         (time, depth, latitude, longitude) float64 46kB ...
    zo         (time, depth, latitude, longitude) float64 46kB ...
Pre-mean shape: (5, 18, 8, 8, 3)
Unscaled shape: (5, 18, 8, 8, 3)
Actual prediction shape: (5, 18, 8, 8, 3)

✅ Saved predicted NetCDF at: E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_Predicted_Actual.nc
✅ Saved true NetCDF at: E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_True_Actual.nc

🎯 DONE — You now have fully physical NetCDF files.


In [ ]:
import numpy as np
import xarray as xr
import os

# =====================================================
# CONFIGURATION
# =====================================================

cleaned_files = [
    r"E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_Predicted_Actual.nc",
    r"E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_True_Actual.nc",
    r"E:\ML Project\WOSC Project\WOSC Dynamic\Yaas_pre_mean.nc"
]

# =====================================================
# LOOP THROUGH FILES
# =====================================================

for nc_path in cleaned_files:
    
    # Extract file name
    name = os.path.basename(nc_path)

    # Open dataset
    ds = xr.open_dataset(nc_path)

    # Extract coordinates
    time_vals = ds.time.values
    depth_vals = ds.depth.values
    lat_vals = ds.latitude.values
    lon_vals = ds.longitude.values

    # Dimensions
    T = len(time_vals)
    D = len(depth_vals)
    H = len(lat_vals)
    W = len(lon_vals)

    print(f"\nFile: {name}")
    print("T, D, H, W =", T, D, H, W)

    print("\nDataset structure:")
    print(ds)

    print("\n" + "="*60)



File: Yaas_Predicted_Actual.nc
T, D, H, W = 5 18 8 8

Dataset structure:
<xarray.Dataset> Size: 138kB
Dimensions:    (time: 5, depth: 18, latitude: 8, longitude: 8)
Coordinates:
  * time       (time) datetime64[ns] 40B 2021-05-23 2021-05-24 ... 2021-05-27
  * depth      (depth) float32 72B 0.0 5.0 10.0 15.0 ... 70.0 80.0 90.0 100.0
  * latitude   (latitude) float32 32B 18.31 18.44 18.56 ... 18.94 19.06 19.19
  * longitude  (longitude) float32 32B 85.44 85.56 85.69 ... 86.06 86.19 86.31
Data variables:
    to         (time, depth, latitude, longitude) float64 46kB ...
    so         (time, depth, latitude, longitude) float64 46kB ...
    zo         (time, depth, latitude, longitude) float64 46kB ...


File: Yaas_True_Actual.nc
T, D, H, W = 5 18 8 8

Dataset structure:
<xarray.Dataset> Size: 138kB
Dimensions:    (time: 5, depth: 18, latitude: 8, longitude: 8)
Coordinates:
  * time       (time) datetime64[ns] 40B 2021-05-23 2021-05-24 ... 2021-05-27
  * depth      (depth) float32 72B 0.0